# Capstone — Content Refresh Opportunity Research Paper

This notebook mirrors the deployed research paper and records the analysis in a reproducible, public-safe form.

## 1. Question

### Research question

Can observed search and engagement signals help prioritize content items for human review as possible refresh opportunities?

### Decision supported

The practical decision is **which content should a reviewer inspect first**. This work does not attempt to prove that a page needs a refresh or that a refresh will improve performance.

The central comparison is between a simple Week-4 rule-based baseline and a Random Forest using independent supporting features.

In [1]:
print('Question and decision framing recorded.')

Question and decision framing recorded.


## 2. Data

### Source and release

The analysis uses the FlyRank pseudonymized warehouse release `flyrank_pseudonymized_warehouse_release_v20260703`. The relevant table is `fact_content_daily_performance`, a daily content-performance panel.

The warehouse contains roughly 79 million daily performance rows overall. For development and model validation, this project uses the March 2026 partition only: `fact_content_daily_performance/month=2026-03/data_0.parquet`.

March was selected as a middle development window rather than the final month so that a later period can remain conceptually sealed for future evaluation.

### Public-safe exclusions

Client names, URLs, raw search queries, and other private identifiers are not included in the paper. Content and client identifiers are shown only in pseudonymous form where needed for reproducibility.

Analytics-unavailable observations are not interpreted as zero activity. The dataset's availability flags are treated as evidence about measurement coverage.

In [1]:
print('March 2026 development window: 9,841,378 rows; 31 columns; 55 clients; 331,437 content items.')

March 2026 development window: 9,841,378 rows; 31 columns; 55 clients; 331,437 content items.


## 3. Methodology

### Unit of analysis

Week 3 established the daily grain as one row per content item, client, and reporting date. No duplicate grain rows were found.

### Baseline

The Week-4 baseline flags a possible refresh opportunity when `gsc_impressions > 100` and `gsc_clicks <= 1`. It is a rule-based decision aid, not ground truth.

### Proxy label

For modeling, the label was defined as: `(gsc_impressions > 100) AND (gsc_clicks <= 1) AND (ga4_sessions == 0)`. The resulting labels were 9,452,885 negatives and 388,493 positives. This is a constructed proxy, not a confirmed refresh outcome.

### Model features

The Random Forest used five features that were not direct components of the proxy-label formula: `gsc_avg_position`, `ga4_pageviews`, `ga4_users`, `ga4_total_engagement_sec`, and `scroll_events`.

### Validation design

The train/test split was grouped by `client_hash_id`: 44 clients in training and 11 in test, with zero client overlap. Week 6 retained the full test set and used a 500,000-row training sample for memory-efficient revalidation.

### Leakage checks

No direct feature-to-label column overlap was detected in the final feature set. The grouped split also passed with zero client overlap. This does not prove that all conceptual dependence has disappeared; some features can still be associated with the search-performance concepts used to construct the proxy label.

### Assumptions

The project assumes that observed performance patterns can be useful for prioritization, while recognizing that measurement availability, client differences, seasonality, SERP changes, and content context can affect those patterns.

In [1]:
print('Baseline: impressions > 100 and clicks <= 1.')
print('Proxy labels: 9,452,885 negative; 388,493 positive.')
print('Grouped split: 44 train clients / 11 test clients / 0 overlap.')
print('Leakage audit: no direct feature-to-label overlap detected.')

Baseline: impressions > 100 and clicks <= 1.
Proxy labels: 9,452,885 negative; 388,493 positive.
Grouped split: 44 train clients / 11 test clients / 0 overlap.
Leakage audit: no direct feature-to-label overlap detected.


## 4. Results (vs baseline)

The baseline was stronger than the Random Forest on the original Week-5 evaluation. The baseline achieved precision **0.768**, recall **1.000**, and F1 **0.868**. The Week-5 Random Forest achieved precision **0.309**, recall **0.693**, and F1 **0.428**.

The ML-09 controlled revalidation produced precision **0.122**, recall **0.973**, and F1 **0.217**. Its high recall came with many false positives, so its F1 was lower than the original Week-5 result.

The key finding is therefore not that ML won. The measured result is that the simple baseline currently provides stronger prioritization under the evaluated setup.

### Feature importance

`gsc_avg_position` accounted for approximately **96.24%** of Random Forest feature importance in Week 5. This identifies ranking position as the strongest measured predictive signal among the selected features; it does not establish causation.

### Result interpretation

The model provides directional decision-support, but the experiment does not justify claiming production-level refresh prediction.

In [1]:
import pandas as pd
results = pd.DataFrame({
    'Approach':['Week-4 Baseline','Week-5 Random Forest','ML-09 Validation RF'],
    'Precision':[0.768,0.309,0.122],
    'Recall':[1.000,0.693,0.973],
    'F1':[0.868,0.428,0.217]
})
display(results)

Approach,Precision,Recall,F1
Week-4 Baseline,0.768,1.000,0.868
Week-5 Random Forest,0.309,0.693,0.428
ML-09 Validation RF,0.122,0.973,0.217


## 5. Limitations

This project has several important limits.

1. The refresh label is a proxy constructed from observed performance conditions; it is not human-confirmed ground truth.
2. March 2026 is one development window. Results may change across time, clients, markets, and content types.
3. The Week-6 model used a 500,000-row training sample and a more constrained Random Forest for memory reasons, so it is a controlled revalidation rather than an identical rerun of Week 5.
4. No causal claim is made. Feature importance and associations do not show that changing a feature will cause better outcomes.
5. The queue is decision-support, not an autonomous content system.
6. Analytics availability can affect observed signals, so missing data must not automatically be interpreted as zero performance.
7. A future study should validate recommendations against human-reviewed outcomes and, ideally, later time periods.

In [1]:
print('Limitation check: proxy label, time window, sampling, causality, measurement coverage, and human validation are documented.')

Limitation check: proxy label, time window, sampling, causality, measurement coverage, and human validation are documented.


## 6. Ranked recommendations

The Week-7 action playbook converts the measured signals into a human-reviewed queue.

### 1 — HIGH_VISIBILITY_LOW_CLICKS
Review pages with more than 100 impressions and no more than 1 click first. Inspect title, snippet, search intent, and content relevance.

### 2 — VISIBLE_LOW_CTR
Review visible pages with CTR below 5%. Inspect SERP presentation and title/snippet relevance. This is a broader supporting signal than the baseline opportunity rule.

### 3 — WEAK_SEARCH_POSITION
For pages with average position above 20, review relevance, content depth, internal linking, and search intent.

### 4 — LOW_EVIDENCE / MONITOR_ONLY
Gather more evidence before investing in a refresh when visibility or ranking evidence is weak.

### Human review and cost/value
Reviewers should check current intent, factual accuracy, business value, SERP context, and expected editing effort before acting. The queue does not estimate financial return.

### No-go cases
Do not automatically publish rewrites, delete or redirect pages, declare content low quality, change search intent, make sensitive claims, or treat the queue score or proxy label as proof of a successful refresh.

The recommended role is prioritization: **what to inspect first, and why**.

In [1]:
print('Week-7 queue: 331,437 content items; 53,994 HIGH; 68,180 MEDIUM; 209,263 LOW.')

Week-7 queue: 331,437 content items; 53,994 HIGH; 68,180 MEDIUM; 209,263 LOW.


## 7. Artifacts the paper embeds

The Week-7 notebook exports the ranked queue to `work/outputs/w07_ranked_action_queue.csv` and the March monitoring reference to `work/outputs/w07_monitoring_reference.json`.

The reproducibility chain is:

- Week 3: data contract and grain validation.
- Week 4: baseline signal audit and rule-based queue.
- Week 5: Random Forest model and baseline comparison.
- Week 6: client-grouped validation, leakage audit, and claim rewrite.
- Week 7: human-reviewed action playbook and ranked queue.
- Week 8: this research paper and deployment.

The repository contains the notebooks and public-safe artifacts needed to inspect the work.

In [1]:
print('Artifacts recorded: w07_ranked_action_queue.csv; w07_monitoring_reference.json.')

Artifacts recorded: w07_ranked_action_queue.csv; w07_monitoring_reference.json.


## Self-check

- [x] Question, data, methodology, results, limitations, recommendations, and artifacts are documented.
- [x] Claims use measured, directional, decision-support language.
- [x] Client names, URLs, and private queries are excluded from the paper.
- [x] Baseline and validation results are reported together.
- [x] Reproducibility links can be added from the deployed page to the repository notebooks.